In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('0.3_scaling_data.csv')

output_dir = 'Assets/'
os.makedirs(output_dir, exist_ok=True)

# SCALING DATA 
pos_cols = ['Work_Hours_Per_Week', 'Screen_Time_Hours', 'Sleep_Hours']
zero_cols = ['Physical_Activity_Hours', 'Meditation_Minutes', 'Coffee_Cups_Per_Day']

df_scaled = df.copy()

print("PROCESSING DATA SCALING...")
# Apply Box-Cox
for col in pos_cols:
    transformed_data, best_lambda = stats.boxcox(df[col])
    df_scaled[col + '_BoxCox'] = transformed_data
    print(f"[{col}] Box-Cox transformation applied (Optimal Lambda = {best_lambda:.4f})")

# Apply Log1p
for col in zero_cols:
    df_scaled[col + '_Log1p'] = np.log1p(df[col])
    print(f"[{col}] Log1p transformation applied")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.set_theme(style="whitegrid")

# Work_Hours_Per_Week (Box-Cox)
sns.histplot(df['Work_Hours_Per_Week'], kde=True, ax=axes[0, 0], color='#4C72B0')
axes[0, 0].set_title('Original: Work Hours Per Week', fontsize=12, fontweight='bold')

sns.histplot(df_scaled['Work_Hours_Per_Week_BoxCox'], kde=True, ax=axes[0, 1], color='#55A868')
axes[0, 1].set_title('Scaled (Box-Cox): Work Hours Per Week', fontsize=12, fontweight='bold')

# Meditation_Minutes (Log1p)
sns.histplot(df['Meditation_Minutes'], kde=True, ax=axes[1, 0], color='#C44E52')
axes[1, 0].set_title('Original: Meditation Minutes', fontsize=12, fontweight='bold')

sns.histplot(df_scaled['Meditation_Minutes_Log1p'], kde=True, ax=axes[1, 1], color='#8172B2')
axes[1, 1].set_title('Scaled (Log1p): Meditation Minutes', fontsize=12, fontweight='bold')

plt.suptitle('Data Distribution Before and After Transformation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

plot_path = os.path.join(output_dir, '0.1_scaling_transformation_EDA.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"\n![Transformation Plot saved at]({plot_path})")
plt.close() 

# EXPORT FILE DATA
output_file = '0.4_scaled_data.csv'
df_scaled.to_csv(output_file, index=False)
print(f"\nScaled data successfully saved to: '{output_file}'")

cols_to_drop = [
    'Work_Hours_Per_Week', 
    'Screen_Time_Hours', 
    'Sleep_Hours', 
    'Physical_Activity_Hours', 
    'Meditation_Minutes', 
    'Coffee_Cups_Per_Day'
]
df_final = df_scaled.drop(columns=cols_to_drop)

# Dummy Variable Trap handling
if 'Chronic_OHE_No' in df_final.columns:
    df_final = df_final.drop(columns=['Chronic_OHE_No'])

y = df_final['Burnout_Risk']
X = df_final.drop(columns=['Burnout_Risk'])

print("\n--- X AND y SPLIT RESULTS ---")
print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFinal features included in the model:")
print(X.columns.tolist())

PROCESSING DATA SCALING...
[Work_Hours_Per_Week] Box-Cox transformation applied (Optimal Lambda = 0.7422)
[Screen_Time_Hours] Box-Cox transformation applied (Optimal Lambda = 0.7477)
[Sleep_Hours] Box-Cox transformation applied (Optimal Lambda = 0.7677)
[Physical_Activity_Hours] Log1p transformation applied
[Meditation_Minutes] Log1p transformation applied
[Coffee_Cups_Per_Day] Log1p transformation applied

![Transformation Plot saved at](Assets/0.1_scaling_transformation_EDA.png)

Scaled data successfully saved to: '0.4_scaled_data.csv'

--- X AND y SPLIT RESULTS ---
Features (X) shape: (50000, 14)
Target (y) shape: (50000,)

Final features included in the model:
['Age', 'Gender', 'Employment_Status', 'Sleep_Quality', 'Stress_Level', 'Occupation', 'Education_Level', 'Chronic_OHE_Yes', 'Work_Hours_Per_Week_BoxCox', 'Screen_Time_Hours_BoxCox', 'Sleep_Hours_BoxCox', 'Physical_Activity_Hours_Log1p', 'Meditation_Minutes_Log1p', 'Coffee_Cups_Per_Day_Log1p']
